<a href="https://colab.research.google.com/github/vitor-thompson/Python-para-financas-investimento-e-analise-de-dados./blob/main/redimento_do_portifolio_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import datetime

# 1. Configurações Iniciais
assets = ['ITUB4.SA','ITSA4.SA','TAEE11.SA','SBSP3.SA','VALE3.SA','PETR4.SA', 'RADL3.SA','WEGE3.SA','BBSE3.SA','RAIL3.SA','B3SA3.SA']
start_analysis = '2025-01-01' # Data de início para o gráfico/análise

# 2. Download Otimizado
print("Baixando dados...")
pf_data = yf.download(assets + ['^BVSP'], start='2020-01-01')['Close']
pf_data = pf_data.fillna(method='ffill') # Preenche lacunas de feriados

# 3. Organização de Compras e Vendas (FIFO)
all_purchases = {
    'ITUB4.SA': [{"data": "2025-12-15", "valor": 40.14, "quantidade": 1}, {"data": "2026-03-16", "valor": 43.07, "quantidade": 5}],
    'ITSA4.SA': [{"data": "2025-12-15", "valor": 11.94, "quantidade": 2}, {"data": "2026-01-15", "valor": 12.09, "quantidade": 1}, {"data": "2026-03-16", "valor": 13.42, "quantidade": 5}],
    'TAEE11.SA': [{"data": "2025-12-15", "valor": 42.99, "quantidade": 2}, {"data": "2026-01-15", "valor": 39.99, "quantidade": 1}, {"data": "2026-02-19", "valor": 44.27, "quantidade": 1}],
    'PETR4.SA': [{"data": "2025-12-15", "valor": 31.91, "quantidade": 3}, {"data": "2026-01-15", "valor": 31.66, "quantidade": 1}, {"data": "2026-02-19", "valor": 37.74, "quantidade": 1}],
    'RADL3.SA': [{"data": "2025-12-15", "valor": 25.34, "quantidade": 4}, {"data": "2026-01-15", "valor": 24.23, "quantidade": 2}, {"data": "2026-02-19", "valor": 26.73, "quantidade": 1}],
    'WEGE3.SA': [{"data": "2025-12-15", "valor": 49.36, "quantidade": 2}, {"data": "2026-01-15", "valor": 46.85, "quantidade": 1}, {"data": "2026-02-19", "valor": 51.62, "quantidade": 1}],
    'BBSE3.SA': [{"data": "2026-03-16", "valor": 34.90, "quantidade": 3}]
}

all_sales = {
    'TAEE11.SA': [{'data': '2026-03-16', 'valor': 42.77, 'quantidade': 2}],
    'PETR4.SA': [{'data': '2026-03-16', 'valor': 45.50, 'quantidade': 2}],
    'WEGE3.SA': [{'data': '2026-03-16', 'valor': 45.62, 'quantidade': 2}],
    'ITUB4.SA': [{'data': '2026-03-17', 'valor': 42.97, 'quantidade': 5}]
}

# 4. Processamento FIFO e Consolidação
portfolio_final = {}

for ticker, purchases in all_purchases.items():
    p_list = sorted(purchases, key=lambda x: x['data'])

    # Aplica vendas se houver
    if ticker in all_sales:
        for sale in all_sales[ticker]:
            qty_to_sell = sale['quantidade']
            while qty_to_sell > 0 and p_list:
                if p_list[0]['quantidade'] <= qty_to_sell:
                    qty_to_sell -= p_list[0]['quantidade']
                    p_list.pop(0)
                else:
                    p_list[0]['quantidade'] -= qty_to_sell
                    qty_to_sell = 0

    # Consolida Posição Atual
    if p_list:
        total_qty = sum(p['quantidade'] for p in p_list)
        avg_price = sum(p['valor'] * p['quantidade'] for p in p_list) / total_qty
        first_date = pd.to_datetime(p_list[0]['data'])
        portfolio_final[ticker] = {'qty': total_qty, 'avg_price': avg_price, 'date': first_date}

# 5. Cálculo de Performance
report = []
for ticker, info in portfolio_final.items():
    current_price = pf_data[ticker].iloc[-1]
    total_invested = info['qty'] * info['avg_price']
    current_value = info['qty'] * current_price
    profit_loss = current_value - total_invested
    perc = (profit_loss / total_invested) * 100

    report.append({
        'Ativo': ticker,
        'Qtd': info['qty'],
        'P. Médio': round(info['avg_price'], 2),
        'Preço Atual': round(current_price, 2),
        'Total Investido': round(total_invested, 2),
        'Valor Atual': round(current_value, 2),
        'Lucro/Prejuízo': round(profit_loss, 2),
        '%': round(perc, 2)
    })

df_report = pd.DataFrame(report).set_index('Ativo')
display(df_report)

# Resumo Total
total_inv = df_report['Total Investido'].sum()
total_atu = df_report['Valor Atual'].sum()
print(f"\n--- RESUMO DO PORTFÓLIO ---")
print(f"Total Investido: R$ {total_inv:.2f}")
print(f"Valor de Mercado: R$ {total_atu:.2f}")
print(f"Performance Total: {((total_atu/total_inv)-1)*100:.2f}%")

Baixando dados...


/tmp/ipykernel_1036/826612582.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  pf_data = yf.download(assets + ['^BVSP'], start='2020-01-01')['Close']
[*********************100%***********************]  12 of 12 completed
/tmp/ipykernel_1036/826612582.py:13: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pf_data = pf_data.fillna(method='ffill') # Preenche lacunas de feriados


,Qtd,P. Médio,Preço Atual,Total Investido,Valor Atual,Lucro/Prejuízo,%
Ativo,,,,,,,
ITUB4.SA,1,43.07,42.28,43.07,42.28,-0.79,-1.83
ITSA4.SA,8,12.88,13.37,103.07,106.96,3.89,3.77
TAEE11.SA,2,42.13,41.85,84.26,83.70,-0.56,-0.66
PETR4.SA,3,33.77,47.00,101.31,141.00,39.69,39.18
RADL3.SA,7,25.22,23.18,176.55,162.26,-14.29,-8.09
WEGE3.SA,2,49.23,46.16,98.47,92.32,-6.15,-6.25
BBSE3.SA,3,34.90,34.52,104.70,103.56,-1.14,-1.09



--- RESUMO DO PORTFÓLIO ---
Total Investido: R$ 711.43
Valor de Mercado: R$ 732.08
Performance Total: 2.90%
